In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from pandas import DataFrame,concat
from sklearn.preprocessing import MinMaxScaler
#TechIndicators
from ta.trend import SMAIndicator,EMAIndicator
from ta.momentum import RSIIndicator
import joblib

In [2]:
data = pd.read_csv("Data/Data-Validation.csv")
data

,Stocks,Date,Open,High,Low,Volume,Close
0,Apple,2005-07-25,1.322084,1.330800,1.314270,294627200,1.316674
1,Apple,2005-07-26,1.322685,1.325690,1.303149,268592800,1.311264
2,Apple,2005-07-27,1.317275,1.324488,1.282412,283749200,1.322083
3,Apple,2005-07-28,1.317876,1.322385,1.301347,251311200,1.316374
4,Apple,2005-07-29,1.309160,1.333805,1.270090,562080400,1.281811
...,...,...,...,...,...,...,...
25135,Johnson&Johnson,2025-07-14,156.869995,157.470001,155.520004,10185600,156.820007
25136,Johnson&Johnson,2025-07-15,156.360001,157.190002,154.800003,6873200,155.169998
25137,Johnson&Johnson,2025-07-16,160.300003,166.119995,159.800003,22134800,164.779999
25138,Johnson&Johnson,2025-07-17,163.179993,164.699997,162.300003,11295700,162.979996


In [3]:
EDA_data = data.copy()

In [4]:
#Encoding
categorical = data.iloc[:,[0]]
OneHot = OneHotEncoder(dtype="int")
Encode = OneHot.fit_transform(categorical)
Encode.toarray()

array([[1, 0, 0, 0, 0],
       [1, 0, 0, 0, 0],
       [1, 0, 0, 0, 0],
       ...,
       [0, 0, 0, 1, 0],
       [0, 0, 0, 1, 0],
       [0, 0, 0, 1, 0]], shape=(25140, 5))

In [5]:
Encode_df = DataFrame(Encode.toarray(),columns=OneHot.get_feature_names_out())
data = concat([Encode_df,data.iloc[:,[1,2,3,4,5,6]]],axis=1)
data

,Stocks_Apple,Stocks_GeneralElectric,Stocks_IBM,Stocks_Johnson&Johnson,Stocks_Microsoft,Date,Open,High,Low,Volume,Close
0,1,0,0,0,0,2005-07-25,1.322084,1.330800,1.314270,294627200,1.316674
1,1,0,0,0,0,2005-07-26,1.322685,1.325690,1.303149,268592800,1.311264
2,1,0,0,0,0,2005-07-27,1.317275,1.324488,1.282412,283749200,1.322083
3,1,0,0,0,0,2005-07-28,1.317876,1.322385,1.301347,251311200,1.316374
4,1,0,0,0,0,2005-07-29,1.309160,1.333805,1.270090,562080400,1.281811
...,...,...,...,...,...,...,...,...,...,...,...
25135,0,0,0,1,0,2025-07-14,156.869995,157.470001,155.520004,10185600,156.820007
25136,0,0,0,1,0,2025-07-15,156.360001,157.190002,154.800003,6873200,155.169998
25137,0,0,0,1,0,2025-07-16,160.300003,166.119995,159.800003,22134800,164.779999
25138,0,0,0,1,0,2025-07-17,163.179993,164.699997,162.300003,11295700,162.979996


In [6]:
#Scaling Each Stocks with MinMaxScaler
Scale = dict()
Stocks = data.columns[0:5]
for i in Stocks:
    df = data[data[i]==1]
    Scaler = MinMaxScaler()
    MinMax = Scaler.fit_transform(df.iloc[:,[6,7,8,9,10]])
    df.iloc[:,[6,7,8,9,10]] = MinMax
    data[data[i]==1] = df
    Scale[i]=Scaler
data  

C:\Users\lipun\AppData\Local\Temp\ipykernel_28020\4251421432.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.0810191  0.07324702 0.07777168 ... 0.00724111 0.00741354 0.00767368]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.iloc[:,[6,7,8,9,10]] = MinMax
C:\Users\lipun\AppData\Local\Temp\ipykernel_28020\4251421432.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.0810191  0.07324702 0.07777168 ... 0.00724111 0.00741354 0.00767368]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  data[data[i]==1] = df


,Stocks_Apple,Stocks_GeneralElectric,Stocks_IBM,Stocks_Johnson&Johnson,Stocks_Microsoft,Date,Open,High,Low,Volume,Close
0,1,0,0,0,0,2005-07-25,0.000176,0.000149,0.000201,0.081019,0.000136
1,1,0,0,0,0,2005-07-26,0.000178,0.000129,0.000157,0.073247,0.000115
2,1,0,0,0,0,2005-07-27,0.000157,0.000125,0.000076,0.077772,0.000157
3,1,0,0,0,0,2005-07-28,0.000159,0.000116,0.000150,0.068088,0.000134
4,1,0,0,0,0,2005-07-29,0.000125,0.000161,0.000028,0.160862,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
25135,0,0,0,1,0,2025-07-14,0.920014,0.914631,0.916164,0.054091,0.914453
25136,0,0,0,1,0,2025-07-15,0.916340,0.912630,0.910968,0.031891,0.902667
25137,0,0,0,1,0,2025-07-16,0.944729,0.976459,0.947055,0.134177,0.971311
25138,0,0,0,1,0,2025-07-17,0.965481,0.966309,0.965098,0.061532,0.958453


<strong>Feature Engineering Per Stocks</strong>

In [7]:
for j in EDA_data["Stocks"].unique():
    df = EDA_data[EDA_data["Stocks"]==j]
    sma = df["Close"].rolling(7).mean()
    df["SMA_7"] = sma
    ema = EMAIndicator(df["Close"],window=7).ema_indicator()
    df["EMA_7"] = ema
    rsi = RSIIndicator(df["Close"],window=7).rsi()
    df["RSI_7"] = rsi
    wvap = ((df["Low"]+df["High"]+df["Close"])/3*(df["Volume"])).cumsum()/df["Volume"].cumsum()
    df["WVAP"] = wvap
    df["Year"] = pd.to_datetime(EDA_data[EDA_data["Stocks"]==j]["Date"]).dt.year
    df["Month"] = pd.to_datetime(EDA_data[EDA_data["Stocks"]==j]["Date"]).dt.month
    if j=="Apple":
        Apple = DataFrame()
        Apple = df.replace(np.nan,0)
        Apple.to_csv(f"Data/{j}",index=False) #Save
    if j=="Microsoft":
        Microsoft = DataFrame()
        Microsoft =df.replace(np.nan,0)
        Microsoft.to_csv(f"Data/{j}",index=False)
    if j=="IBM":
        IBM = DataFrame()
        IBM = df.replace(np.nan,0)
        IBM.to_csv(f"Data/{j}",index=False)
    if j=="GeneralElectric":
        GeneralElectric = DataFrame()
        GeneralElectric = df.replace(np.nan,0)
        GeneralElectric.to_csv(f"Data/{j}",index=False)
    if j=="Johnson&Johnson":
        JohnsonJohnson = DataFrame()  
        JohnsonJohnson = df.replace(np.nan,0)
        JohnsonJohnson.to_csv(f"Data/{j}",index=False)

C:\Users\lipun\AppData\Local\Temp\ipykernel_28020\857438407.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["SMA_7"] = sma
C:\Users\lipun\AppData\Local\Temp\ipykernel_28020\857438407.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["EMA_7"] = ema
C:\Users\lipun\AppData\Local\Temp\ipykernel_28020\857438407.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation:

In [9]:
EDA_data.to_csv("Data/EDA_transformed.csv",index=False)
data.to_csv("Data/Transformed.csv",index=False)

In [10]:
for i in Scale.keys():
    joblib.dump(Scale[i],f"Data/Scaled_{i}.pkl")

<h1><b>FINAL REPORT: DATA-TRANSFORMATION</b></h1>
<li>Encoded Categorical Values "Stocks" using OneHotEncoder</li>
<li>Scaled Numerical Features for each unique "Stocks" using MinMaxScaler for further use in Model Implementations</li>
<li>Performed Feature Engineering on each unique "Stocks" and saved it for further applications , extracted SMA(Simple-Moving avg) using rolling feature, EMA(exponential-Moving avg) using EMAIndicator,RSI value using RSIIndicator and WVAP(Weighted volume avg price) using traditional formula</li>
<li>Saved Scaled Model for each stocks for further uses and applications</li>